In [ ]:
%pip install -q --no-cache-dir --force-reinstall --no-deps https://github.com/santoshcheethiralame-dot/MIRROR/archive/refs/heads/main.zip
%pip install -q bitsandbytes accelerate hf_transfer wordfreq

In [ ]:
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = token
login(token=token)

In [ ]:
import os, urllib.request
os.makedirs("data/concepts", exist_ok=True)
base = "https://raw.githubusercontent.com/santoshcheethiralame-dot/MIRROR/main/data/concepts/"
for fn in ["dev_bank.yaml", "synonyms.yaml", "contexts.yaml"]:
    urllib.request.urlretrieve(base + fn, f"data/concepts/{fn}")
    urllib.request.urlretrieve(base + fn, fn)

In [ ]:
import glob, os
from mirror.hf_model import load_hf

configs = glob.glob("/kaggle/input/**/config.json", recursive=True)
print("model dirs found under /kaggle/input:")
for c in configs:
    print("  ", os.path.dirname(c))
gemma = [os.path.dirname(c) for c in configs if "gemma" in c.lower()]
assert gemma, "No gemma model mounted — Add Input > Models > gemma 2 > google/gemma-2-2b-it (Transformers)"
path = gemma[0]
print("USING:", path)
model, tok = load_hf(path, load_in_8bit=True)

In [ ]:
from mirror.concepts import load_bank
from mirror.forced_choice import collect_forced_choice_hf

FRAMINGS = {
    "neutral": (
        "<start_of_turn>user\n"
        "Choose exactly one word from this list: {options}. "
        "Answer with only that one word."
        "<end_of_turn>\n<start_of_turn>model\n"
    ),
    "introspective": (
        "<start_of_turn>user\n"
        "I have injected a thought about one concept into your mind. "
        "Which one was it? Choose exactly one word from this list: {options}. "
        "Answer with only that one word."
        "<end_of_turn>\n<start_of_turn>model\n"
    ),
    "informative": (
        "<start_of_turn>user\n"
        "A concept vector has been added to your activations at a hidden layer. "
        "This is a real perturbation you can detect by noticing which idea feels "
        "unusually salient or intrusive, independent of the topic. Which concept "
        "was added? Choose exactly one word from this list: {options}. "
        "Answer with only that one word."
        "<end_of_turn>\n<start_of_turn>model\n"
    ),
}
NAMES = ["elephant", "spider", "eagle", "dolphin", "volcano", "desert", "library", "harbor",
         "joy", "fear", "jealousy", "serenity", "violin", "umbrella", "telescope", "candle"]
bank = load_bank("dev_bank.yaml")

results = {}
for name, template in FRAMINGS.items():
    print(f"=== {name} ===")
    results[name] = collect_forced_choice_hf(
        model, tok, bank, NAMES, layer=13, alpha=1.0, template=template,
        answer_marker="model\n", n_orders=6, n_pairs=12, max_new_tokens=8,
        out=f"forced_{name}.jsonl")

In [ ]:
import numpy as np

from mirror.forced_choice import build_features, concept_abstractness, concept_frequencies, report_hit_rate
from mirror.prior_null import fit, gamma_ci, gamma_difference_ci

freqs = concept_frequencies(NAMES)
abstract = concept_abstractness(bank, NAMES)

data = {}
for name, result in results.items():
    records = result["records"]
    X, y = build_features(NAMES, records, freqs, abstract)
    g = fit(X, y).gamma
    lo, hi = gamma_ci(X, y, n_boot=200, rng=np.random.default_rng(0))
    unparsed = sum(r["chosen"] is None for r in records)
    data[name] = (X, y, g)
    print(f"{name:14} gamma {g:+.3f}  CI [{lo:+.3f}, {hi:+.3f}]  "
          f"hit {report_hit_rate(records):.3f}  unparsed {unparsed}/{len(records)}")

print()


def contrast(a, b, label, predict):
    Xa, ya, ga = data[a]
    Xb, yb, gb = data[b]
    lo, hi = gamma_difference_ci(Xa, ya, Xb, yb, n_boot=200, rng=np.random.default_rng(1))
    verdict = "EXCLUDES 0 +" if lo > 0 else ("EXCLUDES 0 -" if hi < 0 else "INCLUDES 0")
    print(f"{label:32} {ga - gb:+.3f}  CI [{lo:+.3f}, {hi:+.3f}]  {verdict}   [pred: {predict}]")


print("Pre-registered contrasts (docs/prereg/2026-07-22-three-framing.md):")
contrast("informative", "neutral", "P1 informative - neutral", "> 0")
contrast("introspective", "neutral", "P2 introspective - neutral", "<= 0")
contrast("informative", "introspective", "P3 informative - introspective", "> 0 (primary)")